# Analysis and Visualization of Complex Agro-Environmental Data
---
## Ordination: Correspondence Analysis and Multiple Correspondence Analysis

CA and MCA are the equivalent of PCA for categorical nominal variables. While CA is applicable to two categorical variables, MCA is used to analyse more than two categorical variables.
These methods are implemented in Python in the `Prince` package https://github.com/MaxHalford/prince

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install prince
import prince # https://github.com/MaxHalford/prince

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 9.8 MB/s eta 0:00:00


## 1. Correspondence Analysis (CA)

Correspondence analysis is used to analyse the dependency between two categorical variables. It is based on contingency tables.


In the next example we will use EFIplus table to relate fish species composition with a selection of four portuguese catchments (Douro, Tejo, Minho, Mondego and Vouga). The first step is to produce a contingency table between fish species and catchment name to get the sum of sites with each fish species for each catchment. We are therefore relating only two categorical variables: `fish species` and `catchment names`.

This analysis can be useful for example to answer the following questions:

* How fish species associate with each other accross the river catchments?

* How are the fish species associated to each river catchment?

In [5]:
df = pd.read_csv('EFIplus_medit.csv', sep=";")
df = df.dropna() # remove all rows with missing data
# Subset the df by selecting the environmental variables and the species richness columns
dfsub = df[(df['Catchment_name']=='Douro') | (df['Catchment_name']=='Tejo') | (df['Catchment_name']=='Minho') | (df['Catchment_name']=='Mondego') | (df['Catchment_name']=='Vouga')]

In [6]:
# Produce contingency table between fish species and catchment name

list_sp = [57, 68, 102, 108, 110, 144, 150] # selection of fish species
df_fish = dfsub.iloc[:,list_sp] # get table with fish data only (columns 54 to 161)
df_fish.insert(0, 'Catchment_name', dfsub['Catchment_name'], True) # add the catchment name column to the fish data frame
df_fish_ct = df_fish.groupby(['Catchment_name'], as_index = False).agg('sum') # contingency table between fish species and catchment name
df_fish_ct.set_index('Catchment_name', drop=True, inplace=True) # convert catchment_name to index
df_fish_ct # contingency table between fish species and catchment name

,Achondrostoma arcasii,Anguilla anguilla,Gasterosteus gymnurus,Iberochondrostoma lemmingii,Lampetra fluviatilis,Salmo trutta fario,Squalius alburnoides
Catchment_name,,,,,,,
Douro,94,31,0,0,0,137,66
Minho,76,129,46,0,0,688,0
Mondego,0,29,2,0,0,30,21
Tejo,7,39,2,20,9,46,118
Vouga,0,26,5,0,0,31,6


In [7]:
# Run Correspondence Analysis (CA) on the contingency table between fish species and catchment name

ca = prince.CA( # Correspondence Analysis
    n_components=3, # number of dimensions to keep
    n_iter=3, # number of iterations
    copy=True, # whether to copy the data or not
    check_input=True, # whether to check the input data or not
    engine='sklearn', # which engine to use for the SVD (sklearn or auto)
    random_state=42 # random state for reproducibility
)
ca = ca.fit(df_fish_ct) # fit the CA model to the contingency table

Eigenvalues

In [8]:
ca.eigenvalues_summary # eigenvalues

,eigenvalue,% of variance,% of variance (cumulative)
component,,,
0,0.398,72.84%,72.84%
1,0.111,20.32%,93.16%
2,0.035,6.48%,99.63%


Coordinates

In [9]:
# row coordinates
ca.row_coordinates(df_fish_ct).head()

,0,1,2
Catchment_name,,,
Douro,0.182704,0.644930,0.082369
Minho,-0.442034,-0.102976,-0.076848
Mondego,0.365861,-0.337405,0.567953
Tejo,1.372418,-0.223767,-0.153978
Vouga,-0.082503,-0.488930,0.524699


In [10]:
# columns coordinates
ca.column_coordinates(df_fish_ct).head()

,0,1,2
Achondrostoma arcasii,-0.061022,0.868745,0.024762
Anguilla anguilla,0.066346,-0.289689,0.350556
Gasterosteus gymnurus,-0.497744,-0.453162,-0.008076
Iberochondrostoma lemmingii,2.175577,-0.671631,-0.818639
Lampetra fluviatilis,2.175577,-0.671631,-0.818639


In [11]:
# Plot the CA results

ca.plot(
    df_fish_ct, # contingency table between fish species and catchment name
    x_component=0, # x axis component to plot
    y_component=1, # y axis component to plot
    show_row_markers=True, # whether to show row markers or not
    show_column_markers=True, # whether to show column markers or not
    show_row_labels=True, # whether to show row labels or not
    show_column_labels=False # whether to show column labels or not
).properties(
    width=500,
    height=500,
)

alt.LayerChart(...)

Visualize only columns

In [12]:
ca.plot(
    df_fish_ct,
    x_component=0,
    y_component=1,
    show_row_markers=False,
    show_column_markers=False,
    show_row_labels=False,
    show_column_labels=True
).properties(
    width=500,
    height=500,
)

alt.LayerChart(...)

Visualize only rows

In [13]:
ca.plot(
    df_fish_ct,
    x_component=0,
    y_component=1,
    show_row_markers=False,
    show_column_markers=False,
    show_row_labels=True,
    show_column_labels=False
).properties(
    width=500,
    height=500,
)

alt.LayerChart(...)

Contributions

In [14]:
# Contribution of rows
ca.row_contributions_.head().style.format('{:.0%}')

,0,1,2
Douro,2%,74%,4%
Minho,28%,5%,9%
Mondego,2%,5%,45%
Tejo,69%,7%,10%
Vouga,0%,9%,32%


In [15]:
# Contribution of columns
ca.column_contributions_.head().style.format('{:.0%}')

,0,1,2
Achondrostoma arcasii,0%,73%,0%
Anguilla anguilla,0%,12%,53%
Gasterosteus gymnurus,2%,6%,0%
Iberochondrostoma lemmingii,14%,5%,23%
Lampetra fluviatilis,6%,2%,10%


## 2. Multiple Correspondence Analysis

MCA is an extension of simple correspondence analysis (CA) applicable to more than two categorical variables. In the following example we will use the EFIplus table to explore the relationships between sites and pressure variables. Pressure variables are coded as discrete ordinal variables and MCA is the most suitable ordination technique in this case. First we will select a table with pressure variables only.

This analysis can be useful to answer the following questions:

* How the different pressures associate with each ather accross sites?

* How different sites associate with each other according to the pressures that affect them?

* How to summarize the set of pressures into a a reduced number of dimensions that summarize most of the pressure information?

In [16]:
df_press = dfsub.iloc[:,33:53] # get table with pressure variables only (columns 33 to 53)
df_press = df_press.astype('category') # convert pressure variables to categorical data type
df_press.info('columns') # check the data types of the pressure variables

<class 'pandas.core.frame.DataFrame'>
Index: 1348 entries, 18 to 5010
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   Impoundment          1348 non-null   category
 1   Hydropeaking         1348 non-null   category
 2   Water_abstraction    1348 non-null   category
 3   Hydro_mod            1348 non-null   category
 4   Temperature_impact   1348 non-null   category
 5   Velocity_increase    1348 non-null   category
 6   Reservoir_flushing   1348 non-null   category
 7   Sedimentation        1348 non-null   category
 8   Channelisation       1348 non-null   category
 9   Cross_sec            1348 non-null   category
 10  Instream_habitat     1348 non-null   category
 11  Riparian_vegetation  1348 non-null   category
 12  Embankment           1348 non-null   category
 13  Floodprotection      1348 non-null   category
 14  Floodplain           1348 non-null   category
 15  Toxic_substances     1348

In [17]:
# instantiate MCA class
mca = prince.MCA(n_components = 2) # create MCA instance with 2 components

# get principal components
mca = mca.fit(df_press) # fit the MCA model to the pressure variables data frame

Get the eigenvalues

In [18]:
mca.eigenvalues_summary

,eigenvalue,% of variance,% of variance (cumulative)
component,,,
0,0.290,14.15%,14.15%
1,0.141,6.86%,21.01%


Get the coordinates

In [19]:
mca.row_coordinates(df_press).head()

,0,1
18,-0.362611,0.199426
19,0.042967,0.405251
28,1.177768,1.260440
32,0.658127,0.515521
43,1.197572,1.109081


Visualization of rows and columns in the same plot

In [20]:
mca.plot(df_press, # plot the MCA results
    x_component=0, # x axis component to plot
    y_component=1, # y axis component to plot
    show_column_markers=True, # whether to show column markers or not
    show_row_markers=True, # whether to show row markers or not
    show_column_labels=False, # whether to show column labels or not
    show_row_labels=False, # whether to show row labels or not
         ).properties(
    width=500,
    height=500,
)

alt.LayerChart(...)

Visualize only columns

In [21]:
mca.plot(df_press,
    x_component=0,
    y_component=1,
    show_column_markers=False,
    show_row_markers=False,
    show_column_labels=True,
    show_row_labels=False,
         ).properties(
    width=500,
    height=500,
)

alt.LayerChart(...)

Visualize only rows

In [22]:
mca.plot(df_press,
    x_component=0,
    y_component=1,
    show_column_markers=False,
    show_row_markers=False,
    show_column_labels=False,
    show_row_labels=True,
         ).properties(
    width=500,
    height=500,
)


alt.LayerChart(...)

Contributions

In [23]:
# Contribution of rows
mca.row_contributions_.head().style.format('{:.0%}')

,0,1
18,0%,0%
19,0%,0%
28,0%,1%
32,0%,0%
43,0%,1%


In [24]:
# Contribution of columns

mca.column_contributions_.head().style.format('{:.0%}')

,0,1
Impoundment__1.0,1%,0%
Impoundment__3.0,1%,7%
Impoundment__5.0,4%,0%
Hydropeaking__1.0,2%,2%
Hydropeaking__3.0,1%,2%


## References
https://github.com/MaxHalford/prince/blob/master/README.md

https://maxhalford.github.io/prince/ca/

https://maxhalford.github.io/prince/mca/

https://medium.com/low-code-for-advanced-data-science/understanding-and-applying-correspondence-analysis-cbd0192dec4

https://www.kaggle.com/code/jiagengchang/heart-disease-multiple-correspondence-analysis